# Tabular Pipeline: Product Recommendation

This notebook cleans and merges the customer social profiles and transactions, explores the merged data, engineers features, and trains a model that predicts which product category a customer will buy. Shared cleaning and feature logic lives in `src/tabular/product_model.py`, so the notebook, the training script and the CLI app all run the same code.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.tabular import product_model as pm

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

social_raw, transactions_raw = pm.load_raw()
print(f"Social profiles: {social_raw.shape}")
print(f"Transactions: {transactions_raw.shape}")

Social profiles: (155, 5)
Transactions: (150, 6)


In [2]:
display(social_raw.head())
display(transactions_raw.head())

,customer_id_new,social_media_platform,engagement_score,purchase_interest_score,review_sentiment
0,A178,LinkedIn,74,4.9,Positive
1,A190,Twitter,82,4.8,Neutral
2,A150,Facebook,96,1.6,Positive
3,A162,Twitter,89,2.6,Positive
4,A197,Twitter,92,2.3,Neutral


,customer_id_legacy,transaction_id,purchase_amount,purchase_date,product_category,customer_rating
0,151,1001,408,2024-01-01,Sports,2.3
1,192,1002,332,2024-01-02,Electronics,4.2
2,114,1003,442,2024-01-03,Electronics,2.1
3,171,1004,256,2024-01-04,Clothing,2.8
4,160,1005,64,2024-01-05,Clothing,1.3


## 2. Data cleaning

### Raw data overview

In [3]:
def overview(name: str, df: pd.DataFrame) -> pd.DataFrame:
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns, {df.duplicated().sum()} exact duplicate rows")
    return pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": df.isna().sum(), "unique": df.nunique()})


display(overview("Social profiles", social_raw))
display(overview("Transactions", transactions_raw))

Social profiles: 155 rows, 5 columns, 5 exact duplicate rows


,dtype,nulls,unique
customer_id_new,str,0,84
social_media_platform,str,0,5
engagement_score,int64,0,50
purchase_interest_score,float64,0,40
review_sentiment,str,0,3


Transactions: 150 rows, 6 columns, 0 exact duplicate rows


,dtype,nulls,unique
customer_id_legacy,int64,0,75
transaction_id,int64,0,150
purchase_amount,int64,0,120
purchase_date,str,0,150
product_category,str,0,5
customer_rating,float64,10,37


In [4]:
duplicates = social_raw[social_raw.duplicated(keep=False)].sort_values("customer_id_new")
display(duplicates)

assert social_raw.duplicated().sum() == 5
assert not transactions_raw.duplicated().any()
assert transactions_raw["transaction_id"].is_unique
assert transactions_raw["customer_rating"].isna().sum() == 10

,customer_id_new,social_media_platform,engagement_score,purchase_interest_score,review_sentiment
18,A116,Twitter,78,1.1,Negative
151,A116,Twitter,78,1.1,Negative
73,A125,LinkedIn,66,4.7,Positive
150,A125,LinkedIn,66,4.7,Positive
76,A155,TikTok,98,4.8,Positive
154,A155,TikTok,98,4.8,Positive
78,A162,Twitter,71,4.8,Positive
153,A162,Twitter,71,4.8,Positive
118,A189,Twitter,59,3.8,Neutral
152,A189,Twitter,59,3.8,Neutral


The social file has 5 exact duplicate rows, one extra copy each for customers A116, A125, A155, A162 and A189, so dropping them loses no information. The transactions file has no duplicates and a unique `transaction_id`, but 10 ratings are missing. The two ID columns describe the same customers in different formats (`A178` and `178`), so both files need a shared integer key before any join.

### Cleaning

In [5]:
social = pm.clean_social(social_raw)
transactions = pm.clean_transactions(transactions_raw)

for name, df in [("Social", social), ("Transactions", transactions)]:
    print(f"{name}: {len(df)} rows, {df[pm.KEY].nunique()} customers, ids {df[pm.KEY].min()} to {df[pm.KEY].max()}")
print("Median rating used for the fill:", transactions_raw["customer_rating"].median())
print("Purchase dates:", transactions["purchase_date"].min().date(), "to", transactions["purchase_date"].max().date())

assert len(social) == 150 and not social.duplicated().any()
assert social[pm.KEY].between(100, 199).all() and transactions[pm.KEY].between(100, 199).all()
assert transactions["purchase_date"].dtype.kind == "M"
assert transactions["customer_rating"].notna().all()
assert transactions["rating_missing"].sum() == 10
assert set(social["social_media_platform"]) <= set(pm.ALLOWED_PLATFORMS)
assert set(social["review_sentiment"]) <= set(pm.ALLOWED_SENTIMENTS)
assert set(transactions[pm.TARGET]) == set(pm.ALLOWED_CATEGORIES)

Social: 150 rows, 84 customers, ids 100 to 199
Transactions: 150 rows, 75 customers, ids 100 to 199
Median rating used for the fill: 3.0
Purchase dates: 2024-01-01 to 2024-05-29


Both files now share an integer `customer_id`, the only join key, and the original ID columns are dropped so nobody can join on them by mistake. Missing ratings are flagged in `rating_missing` and then filled with the median, so the model can still tell which ratings were imputed. The median comes from the full file before the train/test split. That is a small leak, accepted because it is a single value computed from 140 observed ratings. Text columns are stripped and mapped to one canonical spelling, and cleaning raises an error on any value outside the allowed lists.

## 3. Merge with validation

### Why not merge directly

In [6]:
for name, df in [("Social", social), ("Transactions", transactions)]:
    rows_per_customer = df.groupby(pm.KEY).size().value_counts().sort_index().to_dict()
    print(f"{name}, customers by number of rows: {rows_per_customer}")

naive_inner = transactions.merge(social, on=pm.KEY, how="inner")
naive_left = transactions.merge(social, on=pm.KEY, how="left")
print(f"Direct inner join: {len(naive_inner)} rows. Direct left join: {len(naive_left)} rows. Transactions: {len(transactions)}.")
assert (len(naive_inner), len(naive_left)) == (213, 246)

Social, customers by number of rows: {1: 43, 2: 23, 3: 13, 4: 4, 6: 1}
Transactions, customers by number of rows: {1: 30, 2: 26, 3: 12, 4: 4, 5: 2, 6: 1}
Direct inner join: 213 rows. Direct left join: 246 rows. Transactions: 150.


Neither file has one row per customer, so a direct join pairs every transaction with every social row for that customer. That inflates 150 transactions to 213 rows (inner) or 246 rows (left), duplicating labels and overweighting customers with many posts. The fix is to aggregate the social table to one row per customer first, which makes the join many-to-one.

### Aggregate social data per customer

In [7]:
def count_ties(column: str) -> int:
    counts = social.groupby(pm.KEY)[column].value_counts()
    is_top = counts == counts.groupby(level=0).transform("max")
    return int((is_top.groupby(level=0).sum() > 1).sum())


social_agg = pm.aggregate_social(social)
assert social_agg[pm.KEY].is_unique and len(social_agg) == social[pm.KEY].nunique()

print(f"{len(social_agg)} customers with social data")
print(f"Customers with a tie for top platform: {count_ties('social_media_platform')}")
print(f"Customers with a tie for top sentiment: {count_ties('review_sentiment')}")
social_agg.head()

84 customers with social data


Customers with a tie for top platform: 25
Customers with a tie for top sentiment: 15


,customer_id,engagement_score,purchase_interest_score,social_post_count,top_platform,top_sentiment
0,100,77.000000,4.400000,2,Instagram,Negative
1,101,68.000000,1.000000,1,Twitter,Neutral
2,102,51.000000,4.800000,1,LinkedIn,Neutral
3,103,64.333333,2.866667,3,Instagram,Positive
4,104,83.000000,2.933333,3,Twitter,Negative


The aggregate keeps the mean `engagement_score` and `purchase_interest_score`, the number of social rows (`social_post_count`), and the most common platform and sentiment. Many customers have only one or two social rows, so ties for the most common value are frequent (25 for platform, 15 for sentiment). Ties are broken alphabetically. That choice is arbitrary but deterministic, so every rerun builds the same dataset.

### Join strategy

The aggregated social table is left joined onto the 150 transactions on `customer_id`, with `validate="many_to_one"` so pandas raises if the social side ever has more than one row per customer. A left join keeps every transaction, because each one carries a target label and the purchases without social data are still valid. An inner join would discard 33 labeled rows (22 percent of a small dataset) and the model would never learn to score customers without a social profile, which the CLI app can meet. `has_social_data` records which rows matched, so the model can treat missing social signal as information.

In [8]:
merged = pm.merge_social(transactions, social_agg)
stats = pm.validate_merge(merged, transactions)

social_ids, transaction_ids = set(social[pm.KEY]), set(transactions[pm.KEY])
shared_customers = len(social_ids & transaction_ids)

print(f"Rows after merge: {stats['rows']}")
print(f"transaction_id unique: {merged['transaction_id'].is_unique}")
print(f"Nulls in key columns: {int(merged[pm.KEY_COLUMNS].isna().sum().sum())}")
print(f"Matched transactions: {stats['matched']}, unmatched: {stats['unmatched']}")
print(f"Customers in both files: {shared_customers}, social only: {len(social_ids - transaction_ids)}, transactions only: {len(transaction_ids - social_ids)}")

assert stats == {"rows": 150, "matched": 117, "unmatched": 33}
assert merged["transaction_id"].is_unique
assert merged[pm.KEY_COLUMNS].notna().all().all()
assert shared_customers == 61
assert merged.loc[merged["has_social_data"], pm.SOCIAL_SCORES].notna().all().all()
assert merged.loc[~merged["has_social_data"], pm.SOCIAL_SCORES].isna().all().all()
pd.testing.assert_frame_equal(merged, pm.load_and_merge())

Rows after merge: 150
transaction_id unique: True
Nulls in key columns: 0
Matched transactions: 117, unmatched: 33
Customers in both files: 61, social only: 23, transactions only: 14


In [9]:
merged.isna().sum().to_frame("nulls").T

,transaction_id,purchase_amount,purchase_date,product_category,customer_rating,customer_id,rating_missing,engagement_score,purchase_interest_score,social_post_count,top_platform,top_sentiment,has_social_data
nulls,0,0,0,0,0,0,0,33,33,0,0,0,0


All checks pass: the merge keeps exactly one row per transaction, 117 transactions matched a social profile and 33 did not, and 61 customers appear in both files. The 23 social-only customers have no purchases and therefore no label, so they cannot be used for training. Unmatched rows get `"Unknown"` for `top_platform` and `top_sentiment` and 0 for `social_post_count`, which are their true values. The two social scores stay empty and are filled inside the model pipeline with a median fitted on training customers only, so the test split never influences the fill. The last assertion confirms that `pm.load_and_merge()`, which the training script uses, produces exactly this table.